In [1]:
print("hello")

hello


In [6]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

# --- STEP 1: LOAD AND INITIAL PREPROCESSING ---
FILE_NAME = 'energy_data_3_years.csv'
try:
    data = pd.read_csv(FILE_NAME)
except FileNotFoundError:
    print(f"Error: The file '{FILE_NAME}' was not found. Please upload it and use the correct name.")
    exit()

# 1. Ensure Date column is in datetime format
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date').reset_index(drop=True)

print("Data loaded successfully!")
print(f"Date range: {data['Date'].min().date()} to {data['Date'].max().date()}")
print(f"Total records: {len(data)}")
print(f"\nColumns: {list(data.columns)}")

# --- STEP 2: PREPARE DATA FOR FORECASTING ---

# Use Solar Energy as the target variable
target_col = 'Solar_Energy_Produced_kWh'

# Check for missing values
print(f"\nMissing values in {target_col}: {data[target_col].isna().sum()}")

# Fill missing values with forward fill, then backward fill
data[target_col] = data[target_col].fillna(method='ffill').fillna(method='bfill')

# Extract the time series
solar_series = data[target_col].values

print(f"\nTarget variable statistics:")
print(f"  Mean: {solar_series.mean():.2f} kWh")
print(f"  Std: {solar_series.std():.2f} kWh")
print(f"  Min: {solar_series.min():.2f} kWh")
print(f"  Max: {solar_series.max():.2f} kWh")

# --- STEP 3: TRAIN FORECASTING MODELS ---

# Use simple methods that are reliable and don't require complex dependencies
from sklearn.linear_model import LinearRegression
from sklearn.preprocessing import PolynomialFeatures

print("\n" + "="*50)
print("Training Forecasting Models...")
print("="*50)

# Method 1: Exponential Weighted Moving Average (EWMA)
ewma_20 = data[target_col].ewm(span=20).mean()
ewma_60 = data[target_col].ewm(span=60).mean()

# Method 2: Seasonal Decomposition using simple averages
data['day_of_year'] = data['Date'].dt.dayofyear.clip(upper=365)
seasonal_pattern = data.groupby('day_of_year')[target_col].mean()

# Method 3: Linear Trend with Polynomial Features
X = np.arange(len(solar_series)).reshape(-1, 1)
poly_features = PolynomialFeatures(degree=2)
X_poly = poly_features.fit_transform(X)
poly_model = LinearRegression()
poly_model.fit(X_poly, solar_series)

print("✓ Models trained successfully!")

# --- STEP 4: GENERATE FORECASTS FOR 2026 (365 DAYS) ---

MAX_PREDICTION_LENGTH = 365
last_date = data['Date'].max()

# Create future dates for 2026 (Jan 1 - Dec 31, 2026)
forecast_start_date = pd.Timestamp('2026-01-01')
future_dates = pd.date_range(start=forecast_start_date, periods=MAX_PREDICTION_LENGTH, freq='D')
future_data = pd.DataFrame({'Date': future_dates})
future_data['day_of_year'] = future_data['Date'].dt.dayofyear.clip(upper=365)

print(f"\nGenerating {MAX_PREDICTION_LENGTH}-day forecast for 2026...")
print(f"Forecast period: {future_data['Date'].min().date()} to {future_data['Date'].max().date()}")

# Prepare forecast components
ensemble_forecasts = []

# Component 1: EWMA-based forecast (use recent weighted average)
ewma_forecast = np.full(MAX_PREDICTION_LENGTH, ewma_20.iloc[-1])
ensemble_forecasts.append(ewma_forecast)

# Component 2: Seasonal forecast (use day-of-year pattern)
seasonal_forecast = future_data['day_of_year'].map(seasonal_pattern).fillna(seasonal_pattern.mean()).values
ensemble_forecasts.append(seasonal_forecast)

# Component 3: Polynomial trend forecast
X_future = np.arange(len(solar_series), len(solar_series) + MAX_PREDICTION_LENGTH).reshape(-1, 1)
X_future_poly = poly_features.transform(X_future)
trend_forecast = poly_model.predict(X_future_poly)
# Ensure positive values
trend_forecast = np.maximum(trend_forecast, 0)
ensemble_forecasts.append(trend_forecast)

# Component 4: Simple moving average baseline
ma_20 = data[target_col].tail(20).mean()
ma_forecast = np.full(MAX_PREDICTION_LENGTH, ma_20)
ensemble_forecasts.append(ma_forecast)

# Ensemble: Weighted average of all methods
weights = np.array([0.25, 0.35, 0.25, 0.15])  # More weight on seasonal pattern
ensemble_forecast = np.average(ensemble_forecasts, axis=0, weights=weights)

# Smooth the forecast with a rolling average to reduce noise
smoothed_forecast = pd.Series(ensemble_forecast).rolling(window=7, center=True).mean()
smoothed_forecast = smoothed_forecast.fillna(pd.Series(ensemble_forecast)).values

print("✓ Forecast generated successfully!")

# --- STEP 5: CREATE AND SAVE RESULTS ---

# Create forecast DataFrame
forecast_df = future_data[['Date']].copy()
forecast_df['Solar_Energy_Forecast_kWh'] = smoothed_forecast

print("\n" + "="*50)
print(f"| Solar Energy Forecast for 2026 (365 Days) |")
print("="*50)

print(f"\nTotal rows: {len(forecast_df)}")
print(f"\nForecast Summary:")
print(f"  Mean: {smoothed_forecast.mean():.2f} kWh")
print(f"  Std: {smoothed_forecast.std():.2f} kWh")
print(f"  Min: {smoothed_forecast.min():.2f} kWh")
print(f"  Max: {smoothed_forecast.max():.2f} kWh")

print("\nForecast Sample (First 10 days of 2026):")
print(forecast_df.head(10).to_string(index=False))

print("\nForecast Sample (Last 10 days of 2026):")
print(forecast_df.tail(10).to_string(index=False))

# Save the forecast
output_file = 'solar_energy_forecast_2026.csv'
forecast_df.to_csv(output_file, index=False)
print(f"\n✓ Forecast saved to '{output_file}'")
print(f"  Total predictions: {len(forecast_df)} rows")
print(f"  Date range: {forecast_df['Date'].min().date()} to {forecast_df['Date'].max().date()}")

# Create a detailed report by month
print("\n" + "="*50)
print("Forecast Details by Month (2026)")
print("="*50)
forecast_df['Month'] = pd.to_datetime(forecast_df['Date']).dt.to_period('M')
monthly_stats = forecast_df.groupby('Month')['Solar_Energy_Forecast_kWh'].agg(['count', 'mean', 'min', 'max', 'sum'])
monthly_stats.columns = ['Days', 'Mean_kWh', 'Min_kWh', 'Max_kWh', 'Total_kWh']
print(monthly_stats.to_string())

# Summary statistics
print("\n" + "="*50)
print("Total Energy Generation Forecast for 2026")
print("="*50)
total_energy = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"Total Solar Energy (2026): {total_energy:,.2f} kWh")
print(f"Average Daily Production: {smoothed_forecast.mean():,.2f} kWh")
print(f"Peak Day Production: {smoothed_forecast.max():,.2f} kWh")
print(f"Minimum Day Production: {smoothed_forecast.min():,.2f} kWh")

Data loaded successfully!
Date range: 2022-12-10 to 2025-12-08
Total records: 1095

Columns: ['Date', 'Avg_Solar_Irradiance_W_m2', 'Avg_Wind_Speed_m_s', 'Solar_Energy_Produced_kWh', 'Wind_Energy_Produced_kWh']

Missing values in Solar_Energy_Produced_kWh: 0

Target variable statistics:
  Mean: 9.53 kWh
  Std: 2.01 kWh
  Min: 4.50 kWh
  Max: 14.52 kWh

Training Forecasting Models...
✓ Models trained successfully!

Generating 365-day forecast for 2026...
Forecast period: 2026-01-01 to 2026-12-31
✓ Forecast generated successfully!

| Solar Energy Forecast for 2026 (365 Days) |

Total rows: 365

Forecast Summary:
  Mean: 8.33 kWh
  Std: 0.66 kWh
  Min: 6.79 kWh
  Max: 9.31 kWh

Forecast Sample (First 10 days of 2026):
      Date  Solar_Energy_Forecast_kWh
2026-01-01                   7.680817
2026-01-02                   7.340904
2026-01-03                   7.598323
2026-01-04                   7.614906
2026-01-05                   7.589324
2026-01-06                   7.601074
2026-01-07

In [7]:
# Display all 365 rows of forecast data
print("\n" + "="*70)
print("COMPLETE 2026 SOLAR ENERGY FORECAST - ALL 365 DAYS")
print("="*70)
print(f"\nTotal rows: {len(forecast_df)}\n")

# Display all rows without truncation
pd.set_option('display.max_rows', None)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', None)

print(forecast_df.to_string(index=False))

# Reset options
pd.reset_option('display.max_rows')
pd.reset_option('display.max_columns')
pd.reset_option('display.width')
pd.reset_option('display.max_colwidth')

print("\n" + "="*70)
print(f"End of forecast data - All {len(forecast_df)} rows displayed")
print("="*70)


COMPLETE 2026 SOLAR ENERGY FORECAST - ALL 365 DAYS

Total rows: 365

      Date  Solar_Energy_Forecast_kWh   Month
2026-01-01                   7.680817 2026-01
2026-01-02                   7.340904 2026-01
2026-01-03                   7.598323 2026-01
2026-01-04                   7.614906 2026-01
2026-01-05                   7.589324 2026-01
2026-01-06                   7.601074 2026-01
2026-01-07                   7.614656 2026-01
2026-01-08                   7.641571 2026-01
2026-01-09                   7.646486 2026-01
2026-01-10                   7.625899 2026-01
2026-01-11                   7.643811 2026-01
2026-01-12                   7.650223 2026-01
2026-01-13                   7.665634 2026-01
2026-01-14                   7.650711 2026-01
2026-01-15                   7.625786 2026-01
2026-01-16                   7.599195 2026-01
2026-01-17                   7.603769 2026-01
2026-01-18                   7.595176 2026-01
2026-01-19                   7.584081 2026-01
2026-01-20

In [8]:
# --- BONUS: WIND ENERGY FORECAST FOR 2026 ---

print("\n" + "="*70)
print("ADDING WIND ENERGY FORECAST FOR 2026")
print("="*70)

# Use same ensemble method for Wind Energy
wind_col = 'Wind_Energy_Produced_kWh'

# Check if wind column exists in data
if wind_col in data.columns:
    print(f"\n✓ Wind energy data found!")
    
    # Fill missing values
    data[wind_col] = data[wind_col].fillna(method='ffill').fillna(method='bfill')
    wind_series = data[wind_col].values
    
    print(f"\nWind Energy Statistics (Historical):")
    print(f"  Mean: {wind_series.mean():.2f} kWh")
    print(f"  Std: {wind_series.std():.2f} kWh")
    print(f"  Min: {wind_series.min():.2f} kWh")
    print(f"  Max: {wind_series.max():.2f} kWh")
    
    # Train wind energy models
    print(f"\nTraining Wind Energy Models...")
    
    # Method 1: EWMA for wind
    wind_ewma_20 = data[wind_col].ewm(span=20).mean()
    
    # Method 2: Seasonal pattern for wind
    wind_seasonal = data.groupby('day_of_year')[wind_col].mean()
    
    # Method 3: Polynomial trend for wind
    wind_poly_model = LinearRegression()
    wind_poly_model.fit(X_poly, wind_series)
    
    # Method 4: Moving average for wind
    wind_ma_20 = data[wind_col].tail(20).mean()
    
    # Generate wind forecasts
    wind_ensemble = []
    
    # Component 1: EWMA
    wind_ensemble.append(np.full(MAX_PREDICTION_LENGTH, wind_ewma_20.iloc[-1]))
    
    # Component 2: Seasonal
    wind_ensemble.append(future_data['day_of_year'].map(wind_seasonal).fillna(wind_seasonal.mean()).values)
    
    # Component 3: Trend
    wind_trend = wind_poly_model.predict(X_future_poly)
    wind_trend = np.maximum(wind_trend, 0)
    wind_ensemble.append(wind_trend)
    
    # Component 4: MA
    wind_ensemble.append(np.full(MAX_PREDICTION_LENGTH, wind_ma_20))
    
    # Weighted ensemble for wind
    wind_forecast = np.average(wind_ensemble, axis=0, weights=weights)
    
    # Smooth wind forecast
    wind_smoothed = pd.Series(wind_forecast).rolling(window=7, center=True).mean()
    wind_smoothed = wind_smoothed.fillna(pd.Series(wind_forecast)).values
    
    # Add wind forecast to DataFrame
    forecast_df['Wind_Energy_Forecast_kWh'] = wind_smoothed
    
    print("✓ Wind Energy forecast generated!")
    
    # Wind statistics
    print(f"\nWind Forecast Summary (2026):")
    print(f"  Mean: {wind_smoothed.mean():.2f} kWh")
    print(f"  Std: {wind_smoothed.std():.2f} kWh")
    print(f"  Min: {wind_smoothed.min():.2f} kWh")
    print(f"  Max: {wind_smoothed.max():.2f} kWh")
    
else:
    print(f"\n⚠ Wind energy column '{wind_col}' not found in data!")
    print(f"Available columns: {list(data.columns)}")
    print("\nPlease check if the data has a wind energy column.")

# Save combined forecast (Solar + Wind)
combined_file = 'solar_wind_energy_forecast_2026.csv'
forecast_df.to_csv(combined_file, index=False)
print(f"\n✓ Combined forecast saved to '{combined_file}'")

# Display combined forecast summary
print("\n" + "="*70)
print("COMBINED 2026 ENERGY FORECAST SUMMARY")
print("="*70)

total_solar = forecast_df['Solar_Energy_Forecast_kWh'].sum()
print(f"\nTotal Solar Energy (2026): {total_solar:,.2f} kWh")
print(f"Average Daily Solar: {forecast_df['Solar_Energy_Forecast_kWh'].mean():,.2f} kWh")

if 'Wind_Energy_Forecast_kWh' in forecast_df.columns:
    total_wind = forecast_df['Wind_Energy_Forecast_kWh'].sum()
    total_combined = total_solar + total_wind
    
    print(f"\nTotal Wind Energy (2026): {total_wind:,.2f} kWh")
    print(f"Average Daily Wind: {forecast_df['Wind_Energy_Forecast_kWh'].mean():,.2f} kWh")
    
    print(f"\n{'='*70}")
    print(f"COMBINED TOTAL ENERGY (2026): {total_combined:,.2f} kWh")
    print(f"{'='*70}")
    print(f"Solar  : {(total_solar/total_combined)*100:.1f}%")
    print(f"Wind   : {(total_wind/total_combined)*100:.1f}%")
    
    # Monthly combined stats
    print(f"\n{'='*70}")
    print("MONTHLY COMBINED FORECAST (2026)")
    print(f"{'='*70}")
    monthly_combined = forecast_df.groupby('Month')[['Solar_Energy_Forecast_kWh', 'Wind_Energy_Forecast_kWh']].sum()
    monthly_combined['Total_kWh'] = monthly_combined['Solar_Energy_Forecast_kWh'] + monthly_combined['Wind_Energy_Forecast_kWh']
    monthly_combined.columns = ['Solar_kWh', 'Wind_kWh', 'Total_kWh']
    print(monthly_combined.to_string())
    
    # Display first 10 rows with both forecasts
    print(f"\n{'='*70}")
    print("FORECAST SAMPLE (First 10 Days)")
    print(f"{'='*70}")
    print(forecast_df[['Date', 'Solar_Energy_Forecast_kWh', 'Wind_Energy_Forecast_kWh']].head(10).to_string(index=False))



ADDING WIND ENERGY FORECAST FOR 2026

✓ Wind energy data found!

Wind Energy Statistics (Historical):
  Mean: 3.93 kWh
  Std: 3.68 kWh
  Min: 0.00 kWh
  Max: 23.39 kWh

Training Wind Energy Models...
✓ Wind Energy forecast generated!

Wind Forecast Summary (2026):
  Mean: 4.21 kWh
  Std: 1.01 kWh
  Min: 2.92 kWh
  Max: 6.21 kWh

✓ Combined forecast saved to 'solar_wind_energy_forecast_2026.csv'

COMBINED 2026 ENERGY FORECAST SUMMARY

Total Solar Energy (2026): 3,041.66 kWh
Average Daily Solar: 8.33 kWh

Total Wind Energy (2026): 1,537.46 kWh
Average Daily Wind: 4.21 kWh

COMBINED TOTAL ENERGY (2026): 4,579.12 kWh
Solar  : 66.4%
Wind   : 33.6%

MONTHLY COMBINED FORECAST (2026)
          Solar_kWh    Wind_kWh   Total_kWh
Month                                      
2026-01  236.461672   97.512760  333.974431
2026-02  222.359602   85.621674  307.981276
2026-03  258.407285   95.649219  354.056504
2026-04  261.957006   96.926327  358.883332
2026-05  281.610126  114.987647  396.597773
2026-0